# Arena 3D Reconstruction with Gaussian Splatting

Reconstruct a 10-15m arena from 91 photos using 3D Gaussian Splatting on Google Colab.

---
**Workflow:**
1. Install dependencies (COLMAP + 3DGS)
2. Upload images
3. Run COLMAP SfM (or use pre-computed data)
4. Convert data to 3DGS format
5. Train Gaussian Splatting
6. Download the point cloud

**Estimated time:** ~20-30 minutes on a T4 GPU
---

In [ ]:
#@title === 1. Mount Google Drive (for checkpoint saving) ===
import os
from google.colab import drive

DRIVE_PATH = "/content/drive/MyDrive/arena_3dgs"
drive.mount('/content/drive')
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_PATH}")

In [ ]:
#@title === 2. Install Dependencies ===
import os
os.environ['QT_QPA_PLATFORM'] = 'offscreen'
os.environ['DISPLAY'] = ''

%cd /content

# --- System packages (COLMAP) ---
print("[1/4] Installing COLMAP...")
!apt-get update -qq && apt-get install -y -qq colmap
!colmap version

# --- PyTorch with CUDA ---
print("[2/4] Installing PyTorch...")
!pip install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118 -q
import torch
print(f"  PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

# --- Python packages ---
print("[3/4] Installing Python dependencies...")
!pip install plyfile numpy pillow opencv-python-headless tqdm -q

# --- Clone 3DGS repo and install CUDA rasterizer ---
print("[4/4] Installing 3D Gaussian Splatting...")
if not os.path.exists('/content/gaussian-splatting'):
    !git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting -q

%cd /content/gaussian-splatting
!pip install submodules/diff-gaussian-rasterization -q
!pip install submodules/simple-knn -q

# Verify
import diff_gaussian_rasterization
import simple_knn
print("  diff-gaussian-rasterization OK")
print("  simple-knn OK")

print("\nAll dependencies installed!")
%cd /content

---
## Step 3: Upload or Download Images

Choose one of the two options below:
- **Option A**: Upload your own ZIP of images (use if you have new images)
- **Option B**: Download the pre-processed images from GitHub (recommended)

In [ ]:
#@title === 3A: Upload Images (ZIP) ===
from google.colab import files
import zipfile
import os

INPUT_DIR = "/content/gaussian-splatting/input"
os.makedirs(INPUT_DIR, exist_ok=True)

print("Upload your images ZIP file")
uploaded = files.upload()

for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall(INPUT_DIR)
        print(f"Extracted {fname}")

imgs = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"{len(imgs)} images ready at {INPUT_DIR}")

In [ ]:
#@title === 3B: Download Images from GitHub (recommended) ===
import os
import zipfile
import requests

INPUT_DIR = "/content/gaussian-splatting/input"
os.makedirs(INPUT_DIR, exist_ok=True)

GITHUB_REPO = "kaarthik-balakrishnan/arena-3dgs"
"""
# If images are stored as a release asset:
url = f"https://github.com/{GITHUB_REPO}/releases/download/v1.0/images.zip"

# Or if raw images are in the repo:
# For each image, download from raw URL
"""

# Since the images are in the repo as processed files,
# we download individual images from the raw GitHub URL:
import base64
GITHUB_TOKEN = ""  # Leave empty for public repos
headers = {"Authorization": f"token {GITHUB_TOKEN}"} if GITHUB_TOKEN else {}

# Get the list of files in the images directory via GitHub API
api_url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/splat-files-processed"
resp = requests.get(api_url, headers=headers)

if resp.status_code == 200:
    files_list = resp.json()
    for item in files_list:
        if item['name'].lower().endswith(('.jpg', '.jpeg', '.png')):
            img_resp = requests.get(item['download_url'], headers=headers)
            with open(os.path.join(INPUT_DIR, item['name']), 'wb') as f:
                f.write(img_resp.content)
    imgs = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Downloaded {len(imgs)} images from GitHub")
else:
    print(f"GitHub API returned {resp.status_code}. Use Option 3A (manual upload) instead.")
    print("Or check if the repo is public and the path is correct.")

imgs = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Total: {len(imgs)} images")
if len(imgs) > 0:
    print(f"Sample: {imgs[0]}")

---
## Step 4: Run COLMAP (Structure from Motion)

This step estimates camera poses and a sparse 3D point cloud from the images.

**Test mode (recommended first):** Run COLMAP on just ~10 images to verify everything works. Then run on all 91 images.

**Checkpoint:** COLMAP output is saved to Google Drive so you can resume if disconnected.

In [ ]:
#@title === 4A: Test Run COLMAP on 10 Images ===
# Run COLMAP on a small subset first to verify the pipeline works.
# If this succeeds, proceed to 4B for the full run.

import os
import shutil
from pathlib import Path

os.environ['QT_QPA_PLATFORM'] = 'offscreen'

INPUT_DIR = "/content/gaussian-splatting/input"
TEST_DIR = "/content/gaussian-splatting/test_input"
TEST_OUT = "/content/gaussian-splatting/test_sparse"

# Select first 10 images for test
all_imgs = sorted([f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
test_imgs = all_imgs[:10]

os.makedirs(TEST_DIR, exist_ok=True)
for img in test_imgs:
    shutil.copy2(os.path.join(INPUT_DIR, img), os.path.join(TEST_DIR, img))
print(f"Copied {len(test_imgs)} test images")

# Run COLMAP on test set
# Note: GPU flags omitted for cross-version compatibility.
# Colab's COLMAP uses GPU by default when CUDA is available.
!mkdir -p {TEST_OUT}
!colmap feature_extractor \
    --database_path {TEST_OUT}/database.db \
    --image_path {TEST_DIR} \
    --ImageReader.single_camera 1

!colmap exhaustive_matcher \
    --database_path {TEST_OUT}/database.db

!colmap mapper \
    --database_path {TEST_OUT}/database.db \
    --image_path {TEST_DIR} \
    --output_path {TEST_OUT}

# Check result
if os.path.exists(f"{TEST_OUT}/0"):
    print(f"\nTest COLMAP succeeded! Output in {TEST_OUT}/0")
    !ls {TEST_OUT}/0/
else:
    print(f"COLMAP test failed - check terminal output above")

In [ ]:
#@title === 4B: Full COLMAP on All Images ===
# Run COLMAP on all images to get complete camera poses.
# Expected output: ~30 registered images in sparse/0/
#
# NOTE: If you already have COLMAP output (from the GitHub repo),
# skip this cell and go to Step 5.

import os
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

INPUT_DIR = "/content/gaussian-splatting/input"
COLMAP_DIR = "/content/gaussian-splatting/sparse"
!mkdir -p {COLMAP_DIR}

# Check for checkpoint
CHECKPOINT_DB = f"{DRIVE_PATH}/database.db"
if os.path.exists(CHECKPOINT_DB):
    print("Found checkpoint database, copying...")
    !cp {CHECKPOINT_DB} {COLMAP_DIR}/database.db

# Step 1: Feature extraction
print("\n=== Step 1: Feature Extraction ===")
# Note: GPU flags omitted for cross-version COLMAP compatibility.
# Colab uses GPU by default when CUDA is available.
!colmap feature_extractor \
    --database_path {COLMAP_DIR}/database.db \
    --image_path {INPUT_DIR} \
    --ImageReader.single_camera 1 \
    --SiftExtraction.max_num_features 8192

# Save checkpoint to Drive
!cp {COLMAP_DIR}/database.db {DRIVE_PATH}/database.db
print("\nCheckpoint saved to Drive")

# Step 2: Feature matching
print("\n=== Step 2: Feature Matching ===")
!colmap exhaustive_matcher \
    --database_path {COLMAP_DIR}/database.db
!cp {COLMAP_DIR}/database.db {DRIVE_PATH}/database.db
print("\nCheckpoint saved to Drive")

# Step 3: Sparse reconstruction
print("\n=== Step 3: Sparse Reconstruction ===")
!colmap mapper \
    --database_path {COLMAP_DIR}/database.db \
    --image_path {INPUT_DIR} \
    --output_path {COLMAP_DIR}

# Copy results to Drive
!cp -r {COLMAP_DIR} {DRIVE_PATH}/sparse_backup

# Show result
if os.path.exists(f"{COLMAP_DIR}/0"):
    !ls {COLMAP_DIR}/0/
    print("\nCOLMAP completed!")
else:
    print("COLMAP did not produce output. Check logs above.")

---
## Step 5: Use Pre-computed COLMAP Data

COLMAP has already been run locally. This cell downloads the pre-computed COLMAP output (cameras.txt, images.txt, points3D.txt) from GitHub so you can skip the 10-minute COLMAP step above.

In [ ]:
#@title === 5: Download Pre-computed COLMAP Data ===
# Skip COLMAP (Step 4) and use our pre-computed camera poses.
# Saves directly to the location expected by the 3DGS trainer.

import os
import requests

INPUT_DIR = "/content/gaussian-splatting/input"
SPARSE_DIR = os.path.join(INPUT_DIR, "sparse", "0")
os.makedirs(SPARSE_DIR, exist_ok=True)

GITHUB_REPO = "kaarthik-balakrishnan/arena-3dgs"
BASE_URL = f"https://raw.githubusercontent.com/{GITHUB_REPO}/main"

files_to_download = [
    "colmap_data/cameras.txt",
    "colmap_data/images.txt",
    "colmap_data/points3D.txt",
]

local_names = ["cameras.txt", "images.txt", "points3D.txt"]

for remote, local in zip(files_to_download, local_names):
    url = f"{BASE_URL}/{remote}"
    print(f"Downloading {local}...")
    r = requests.get(url)
    if r.status_code == 200:
        with open(os.path.join(SPARSE_DIR, local), 'w') as f:
            f.write(r.text)
        print(f"  OK ({len(r.text)/1024:.0f} KB)")
    else:
        print(f"  FAILED (status {r.status_code})")

# Verify
all_ok = True
for f in local_names:
    path = os.path.join(SPARSE_DIR, f)
    if os.path.exists(path):
        lines = sum(1 for _ in open(path))
        print(f"  {f}: {lines} lines")
    else:
        print(f"  {f}: NOT FOUND")
        all_ok = False

if all_ok:
    print("\nPre-computed COLMAP data ready at:", SPARSE_DIR)
else:
    print("\nSome files failed to download. Check the status above.")

---
## Step 6: Convert COLMAP to 3DGS Format

The official 3DGS repo needs the data in a specific format. This step converts the COLMAP output (or the downloaded pre-computed data) into the expected structure.

In [ ]:
#@title === 6: Convert Data to 3DGS Format ===
%cd /content/gaussian-splatting

INPUT_DIR = "/content/gaussian-splatting/input"
SPARSE_DIR = os.path.join(INPUT_DIR, "sparse", "0")

import os
import glob

# The 3DGS trainer expects: input/images/*.jpg and input/sparse/0/*.txt
# Move images into an images/ subdirectory if they're scattered in input/
images_dir = os.path.join(INPUT_DIR, "images")
if not os.path.exists(images_dir):
    os.makedirs(images_dir, exist_ok=True)
    for ext in ['*.jpg', '*.jpeg', '*.png']:
        for f in glob.glob(os.path.join(INPUT_DIR, ext)):
            os.rename(f, os.path.join(images_dir, os.path.basename(f)))
    print(f"Moved {len(os.listdir(images_dir))} images to input/images/")

# Verify the COLMAP files are in the right place
required = ["cameras.txt", "images.txt", "points3D.txt"]
missing = [f for f in required if not os.path.exists(os.path.join(SPARSE_DIR, f))]

if missing:
    print(f"ERROR: Missing files in {SPARSE_DIR}: {missing}")
    print("Run Step 5 (Download Pre-computed COLMAP Data) first.")
else:
    !ls -lh {SPARSE_DIR}/
    !ls {images_dir} | head -5
    print(f"  ... ({len(os.listdir(images_dir))} total)")
    
    # Count registered images
    with open(os.path.join(SPARSE_DIR, "images.txt")) as f:
        img_lines = [l for l in f if l.strip() and not l.startswith('#')]
        num_images = len(img_lines) // 2
        print(f"\nCOLMAP data: {num_images} registered images")
    
    # Check images exist
    available = set(os.listdir(images_dir))
    missing_imgs = []
    with open(os.path.join(SPARSE_DIR, "images.txt")) as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.strip().split()
            if len(parts) >= 10:
                img_name = parts[9]
                if img_name not in available:
                    missing_imgs.append(img_name)
    
    if missing_imgs:
        print(f"WARNING: {len(missing_imgs)} images referenced in COLMAP but not found:")
        for m in missing_imgs[:5]:
            print(f"  - {m}")
    else:
        print("All referenced images found! Ready for training.")

---
## Step 7: Train 3D Gaussian Splatting

This is the main training step. It runs the official 3DGS training script.

**Settings:**
- 7000 iterations (good quality, ~15 min on T4)
- 3000 iterations (quick, ~7 min on T4)
- Checkpoints saved every 500 iterations

In [ ]:
#@title === 7A: Quick Training (3000 iterations, ~7 min) ===
%cd /content/gaussian-splatting

INPUT_DIR = "/content/gaussian-splatting/input"

# The official 3DGS training expects:
# -s points to the directory containing images/ and sparse/

!python train.py \
    -s {INPUT_DIR} \
    --iterations 3000 \
    --checkpoint 500 \
    --test_iterations 3000 \
    --save_iterations 3000 \
    --quiet

print("\nTraining complete! Run the next cell to find results.")

In [ ]:
#@title === 7B: Full Quality Training (7000 iterations, ~15 min) ===
%cd /content/gaussian-splatting

INPUT_DIR = "/content/gaussian-splatting/input"

!python train.py \
    -s {INPUT_DIR} \
    --iterations 7000 \
    --checkpoint 500 \
    --test_iterations 7000 \
    --save_iterations 7000 \
    --quiet

print("\nTraining complete! Run the next cell to find results.")

---
## Step 8: Export & Download Results

Extract the trained point cloud and save it to Google Drive, then download.

In [ ]:
#@title === 8: Export and Download Point Cloud ===
import os
from google.colab import files

# Find the trained point cloud
MODEL_DIR = "/content/gaussian-splatting/output/point_cloud"
if not os.path.exists(MODEL_DIR):
    # Try alternative paths
    MODEL_DIR = "/content/gaussian-splatting/output"

# Find the latest iteration
iter_dirs = [d for d in os.listdir(MODEL_DIR) if d.startswith('iteration_')]
if iter_dirs:
    latest = sorted(iter_dirs)[-1]
    pc_path = os.path.join(MODEL_DIR, latest)
    print(f"Latest iteration: {latest}")
else:
    pc_path = MODEL_DIR
    print(f"Looking in: {pc_path}")

# Find PLY file
ply_files = [f for f in os.listdir(pc_path) if f.endswith('.ply')]
if ply_files:
    src = os.path.join(pc_path, ply_files[0])
    dst = "/content/arena_3dgs_pointcloud.ply"
    !cp "{src}" "{dst}"
    !ls -lh "{dst}"
    print(f"\nPoint cloud ready: {dst}")
    
    # Copy to Google Drive
    !cp "{dst}" "{DRIVE_PATH}/arena_3dgs_pointcloud.ply"
    print(f"Backed up to Drive: {DRIVE_PATH}/arena_3dgs_pointcloud.ply")
    
    # Download
    print("\nDownloading to your computer...")
    files.download(dst)
else:
    print(f"No PLY files found in {pc_path}")
    !ls -R {MODEL_DIR} 2>/dev/null

---
## Troubleshooting

| Problem | Solution |
|---------|----------|
| **CUDA out of memory** | Run `7A` (3000 iterations) instead of `7B` |
| **COLMAP crashes** | Use pre-computed COLMAP data (Step 5) |
| **Session disconnects** | Results are saved to Google Drive, resume from Step 6 |
| **Poor quality** | Run `7B` (7000 iterations) for better results |

**Viewing the .ply file:**
- [MeshLab](https://www.meshlab.net/) (free, desktop)
- [CloudCompare](https://www.cloudcompare.org/) (free, desktop)
- [Potree](https://potree.org/) (web-based)

---